# 2. Open Queuing network simulation (8 pt)
The second part of the project is to make an open-queuing network simulator, where the jobs arrive from the outside following Poisson processes with the rate of lambda. The system is shown as the following figure:

![image](Figure.png)

Each job needs to go to the CPU, following an exponential distribution with a mean of 10 jobs per second, and then proceeds by either one of the disks. There are two disks, each of which has an exponentially-distributed processing time. The fast disk has a processing rate of 12 jobs per second and the slow disk has the rate of 9 jobs per second. The number of buffer spaces in each queue is infinite.


# Questions:
(4pt) Use your simulator to find the maximum sustainable throughput of such a system in terms of number of jobs. Support your answer with simulation results.

(4pt) Find out the average response times for following cases over three arrival rates of your choice: case (i) - a single queue in front of fast and slow disk; case (ii) - a separate queue for each disk, jointly applying the shortest queue load balancing strategy when sending the jobs from the CPU to disk.

![image](Figure_again.png)


In [5]:
import simpy
import numpy as np

class OpenNetwork:
    def __init__(self, env, cpu_rate, slow_rate, fast_rate):
        self.env = env
        self.cpu = simpy.Resource(env, capacity=1)
        self.slow_disk = simpy.Resource(env, capacity=1)
        self.fast_disk = simpy.Resource(env, capacity=1)
        self.cpu_rate = cpu_rate
        self.slow_rate = slow_rate
        self.fast_rate = fast_rate
        self.done = 0

    def job(self):
        with self.cpu.request() as req:
            yield req
            yield self.env.timeout(np.random.exponential(1/self.cpu_rate))
        # Pick disk at random
        if np.random.random() < 0.5:
            with self.slow_disk.request() as req:
                yield req
                yield self.env.timeout(np.random.exponential(1/self.slow_rate))
        else:
            with self.fast_disk.request() as req:
                yield req
                yield self.env.timeout(np.random.exponential(1/self.fast_rate))
        self.done += 1

def generate_jobs(env, system, lam):
    while True:
        yield env.timeout(np.random.exponential(1/lam))
        env.process(system.job())

def simulate(lam, sim_time=10000):
    env = simpy.Environment()
    system = OpenNetwork(env, cpu_rate=10, slow_rate=9, fast_rate=12)
    env.process(generate_jobs(env, system, lam))
    env.run(until=sim_time)
    return system.done / sim_time

# Test a range of λ
results = []
for lam in np.arange(5, 15, 1):
    avg_throughput = np.mean([simulate(lam) for _ in range(3)])
    results.append((lam, avg_throughput))
print(results)


[(np.int64(5), np.float64(5.008666666666667)), (np.int64(6), np.float64(6.013466666666666)), (np.int64(7), np.float64(6.998233333333332)), (np.int64(8), np.float64(7.982266666666667)), (np.int64(9), np.float64(9.003433333333334)), (np.int64(10), np.float64(9.9624)), (np.int64(11), np.float64(10.004333333333333)), (np.int64(12), np.float64(9.975266666666666)), (np.int64(13), np.float64(9.995866666666666)), (np.int64(14), np.float64(10.0057))]
